# Business understanding
(to give quick context inccase the document cant be opent with the full context)

Increasing heatwaves create unequal heat stress across neighbourhoods. The Cool Neighbourhoods project aims to identify where and when heat stress occurs, both indoors and outdoors. The business objective is to help municipalities understand differences between neighbourhoods and make better-informed decisions about where cooling measures, such as urban greening, are most needed. Machine learning will be used to identify patterns and classify or predict heat stress, supporting decision-making and protecting vulnerable residents.

<hr>

# Set-up
### import libary

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import glob
import re

Matplotlib is building the font cache; this may take a moment.


### load and merge datasets

note:
merging still needs to be done but im currently very confused about it, im guessing the datatests we got are from 8 different sensors -Naïmi


In [3]:
all_files = glob.glob("data/*.csv")
df_list = []

#concat data from all files and add column with source
for filename in all_files:
    temp_df = pd.read_csv(filename, sep=";")
    sensor_name = filename.split("/")[-1].split(".")[0].split("\\")[-1]
    temp_df["sensor_ID"] = sensor_name
    df_list.append(temp_df)

df = pd.concat(df_list, ignore_index=True)

#change strings into floats
if "BatteryVoltage" in df.columns:
    df["BatteryVoltage"] = (
        df["BatteryVoltage"]
        .astype(str)
        .str.replace(",", ".", regex=False)
        .str.extract(r"([\d\.]+)")[0] #strip [V] from the string
        .astype(float)
    )

if "Pressure1" in df.columns:
    df["Pressure1"] = (
      df["Pressure1"]
      .astype(str)
      .str.replace(",", ".", regex=False)
      .str.extract(r"([\d\.]+)")[0] #strip [nbar] from the string
      .astype(float)
)
  
df.head()

print("Dataset Quantification:")
print(f"- Total Observations (Rows): {df.shape[0]}")
print(f"- Total Variables (Columns): {df.shape[1]}")
print(f"- Total Cells: {df.size}")
print(f"- Total Missing Values: {df.isnull().sum().sum()}")
print(f"- Total Duplicates: {df.duplicated().sum()}")
print("\nVariable Types & Data Types:")
print(df.dtypes)

numeric_cols = df.select_dtypes(include=["number"]).columns
summary_metrics = pd.DataFrame(
    {
        "Min": df[numeric_cols].min(),
        "Max": df[numeric_cols].max(),
        "Mean": df[numeric_cols].mean(),
        "Median": df[numeric_cols].median(),
        "Unique Values": df[numeric_cols].nunique(),
    }
)
print("\nNumeric Summary (Min, Max, Mean, Median, Unique):")
print(summary_metrics)


Dataset Quantification:
- Total Observations (Rows): 32000
- Total Variables (Columns): 6
- Total Cells: 192000
- Total Missing Values: 0
- Total Duplicates: 0

Variable Types & Data Types:
Message Timestamp        str
BatteryVoltage       float64
Humidity1            float64
Pressure1            float64
Temperature1         float64
sensor_ID                str
dtype: object

Numeric Summary (Min, Max, Mean, Median, Unique):
                   Min      Max         Mean   Median  Unique Values
BatteryVoltage    2.74     3.33     3.048795     3.02             24
Humidity1        18.27   100.00    69.952707    71.36           6315
Pressure1       997.30  1033.60  1017.934900  1018.40            364
Temperature1      4.66    40.58    19.472076    19.20           2920


<hr>

# Data Understanding


* Identify the dataset format (structured, unstructured or semi-structured).
* Identify its arrangement (DataFrame, matrix, list or vector).
* Investigate how missing values are represented (NULL, NA, blanks or none).
* List all variables (columns).
* Classify each variable as alphabetic, numeric or alphanumeric.
* Identify its generic data type:
    - Nominal
    - Ordinal
    - Discrete
    - Continuous
    - Boolean
    - Date
* Investigate whether the data was collected through random or multiple sampling.
* Calculate the total number of cells. (rows times columns)
* Look into the Central Tendency

In [4]:
# Rows and numbers of colums
df.shape

(32000, 6)

In [5]:
# Amount of cells
df.size

192000

In [6]:
# Variables and technical data types
df.dtypes

Message Timestamp        str
BatteryVoltage       float64
Humidity1            float64
Pressure1            float64
Temperature1         float64
sensor_ID                str
dtype: object

In [7]:
# Central tendency
df.describe()

,BatteryVoltage,Humidity1,Pressure1,Temperature1
count,32000.000000,32000.000000,32000.000000,32000.000000
mean,3.048795,69.952707,1017.934900,19.472076
std,0.082154,14.483959,6.507201,5.571443
min,2.740000,18.270000,997.300000,4.660000
25%,3.010000,60.730000,1014.400000,15.720000
50%,3.020000,71.360000,1018.400000,19.200000
75%,3.040000,80.860000,1022.300000,22.840000
max,3.330000,100.000000,1033.600000,40.580000


In [8]:
# For also the categorical data in the table
df.describe(include="all")

,Message Timestamp,BatteryVoltage,Humidity1,Pressure1,Temperature1,sensor_ID
count,32000,32000.000000,32000.000000,32000.000000,32000.000000,32000
unique,31884,NaN,NaN,NaN,NaN,8
top,"2026-07-18, 15:55:02",NaN,NaN,NaN,NaN,middelburg - 005
freq,2,NaN,NaN,NaN,NaN,4249
mean,NaN,3.048795,69.952707,1017.934900,19.472076,NaN
std,NaN,0.082154,14.483959,6.507201,5.571443,NaN
min,NaN,2.740000,18.270000,997.300000,4.660000,NaN
25%,NaN,3.010000,60.730000,1014.400000,15.720000,NaN
50%,NaN,3.020000,71.360000,1018.400000,19.200000,NaN
75%,NaN,3.040000,80.860000,1022.300000,22.840000,NaN


In [9]:
df.nunique()

Message Timestamp    31884
BatteryVoltage          24
Humidity1             6315
Pressure1              364
Temperature1          2920
sensor_ID                8
dtype: int64

## Data shapes (graphs)

* Create a distribution plot (histogram).
* Describe whether the distribution is:
    - Unimodal or multimodal.
    - Symmetrical, uniform, left-skewed or right-skewed.
    - Relatively steep or flat.
* Data Volatility
* Boxplots
* correlation matrix
* scatter plots

## Data preperation

* Check and correct measurement errors.
* Investigate and handle outliers.
* Make sure timestamps and dates are correctly formatted.
* New variables (examples: Hour of the day. Month or season. Day/night indicator. Heatwave indicator. Temperature differences between indoor and outdoor sensors.Greenery or paving indicators, if the required data is available.)
* Create target variable
* Identify categorical variables that need encoding.
* Apply one-hot encoding to nominal variables.
* Apply ordinal encoding to ordered categories.
* Use binary encoding for variables with two possible values, where appropriate.

## Data Cleaning

* Count missing values.
* Find duplicates
* FIx them if fixes are needed

<hr>